# 3-1 棒グラフ

ここから第3章。pandas で集計したデータを **グラフで可視化** する章です。

「商品別の売上を棒グラフで」「月ごとの推移を折れ線で」というのは、レポートの定番。Excel ならグラフボタンですが、Python では **matplotlib** というライブラリを使って 3〜5 行で書きます。

3-1 では最も使う **棒グラフ** から始めます。

## このノートのゴール

- `matplotlib.pyplot` (`plt`) の基本の流れ（`plt.bar` → `plt.show`）が分かる
- グラフに **タイトル・軸ラベル** を付けられる
- **縦棒 (`bar`) / 横棒 (`barh`)** を使い分けられる
- **サイズ・色** を指定して見栄えを整えられる
- 2-3, 2-5 で習った **`groupby` + `sort_values`** の結果をそのまま棒グラフにできる

## matplotlib とは

**matplotlib（マットプロットリブ）** は Python で **グラフを描くための定番ライブラリ** です。Excel のグラフ機能と同じ役割を果たします。

| やりたいこと | Excel | matplotlib |
|---|---|---|
| 棒グラフ | 「グラフの挿入」→「縦棒」 | `plt.bar(x, y)` |
| 横棒グラフ | 「グラフの挿入」→「横棒」 | `plt.barh(x, y)` |
| タイトル | グラフ要素 → グラフタイトル | `plt.title("...")` |
| 軸ラベル | グラフ要素 → 軸ラベル | `plt.xlabel("...")` / `plt.ylabel("...")` |
| サイズ変更 | グラフ枠をドラッグ | `plt.figure(figsize=(横, 縦))` |

**呼び方の慣習**: `import matplotlib.pyplot as plt` で読み込み、コードでは `plt.bar(...)` のように書きます。これは pandas の `pd` と同じく世界共通のお作法です。

## 準備: 日本語が出るようにする

matplotlib のデフォルトでは **日本語が「□□□」のように文字化け** します（フォントが入っていないため）。Colab で日本語を出すには、**`japanize-matplotlib`** という補助ライブラリを入れるのが一番簡単です。

**1回 `pip install` するだけで、後は import するだけで日本語が出るようになります。**

> 💡 フォント周りの詳細は **3-4** で扱います。ここでは「**呪文として最初に書いておく**」で OK です。

In [ ]:
# 日本語フォントの準備（最初の1回だけ実行すれば十分）
!pip install -q japanize-matplotlib

import matplotlib.pyplot as plt
import japanize_matplotlib  # ← import するだけで日本語フォントに切り替わる
import pandas as pd

## 1. 最小限の棒グラフ

**たった2行** で棒グラフが描けます。

```python
plt.bar(x軸の値, y軸の値)
plt.show()
```

- `plt.bar(...)` で棒グラフを **準備**
- `plt.show()` で **画面に出す**

> Colab では `plt.show()` を書かなくても表示されますが、**書く方が安全**（複数のグラフを並べたいときの暗黙の区切りになる）。

In [ ]:
# 商品名と売上のリスト（まずは手書きで小さく試す）
products = ["りんご", "ばなな", "みかん", "メロン", "ぶどう"]
sales    = [12000, 8500, 6800, 28000, 9800]

plt.bar(products, sales)
plt.show()

## 2. タイトル・軸ラベルを付ける

グラフは **「何を表しているか」が一目で分かる** ようにタイトルとラベルを付けるのが鉄則です。

| 関数 | 何を設定するか |
|---|---|
| `plt.title("...")` | グラフ全体のタイトル |
| `plt.xlabel("...")` | x 軸（横軸）の名前 |
| `plt.ylabel("...")` | y 軸（縦軸）の名前 |

In [ ]:
plt.bar(products, sales)
plt.title("商品別 売上")
plt.xlabel("商品")
plt.ylabel("売上 (円)")
plt.show()

## 3. 横棒グラフ — `plt.barh`

**項目名が長いとき**（部署名、顧客名、本のタイトルなど）は、縦棒だとラベルが斜めになって読みにくくなります。そんなときは **横棒グラフ** が便利です。

書き方は `bar` を `barh` に変えるだけ（h = horizontal、水平）。**x と y の役割が入れ替わる** ので、`xlabel` / `ylabel` も入れ替えます。

In [ ]:
plt.barh(products, sales)
plt.title("商品別 売上")
plt.xlabel("売上 (円)")   # ← 横軸が数値になる
plt.ylabel("商品")          # ← 縦軸が項目になる
plt.show()

## 4. サイズ・色を整える

デフォルトのグラフは少し小さいので、レポート用に **サイズを指定** することがほとんどです。色も指定できます。

| 設定 | コード |
|---|---|
| サイズ（インチ） | `plt.figure(figsize=(横, 縦))` を **`plt.bar` の前** に書く |
| 棒の色 | `plt.bar(..., color="色名")` |
| 棒の幅（縦棒のみ） | `plt.bar(..., width=0.5)` |

色名は `"red"` `"blue"` `"green"` `"orange"` `"gray"` などの英語名や、`"#1f77b4"` のような HEX 指定が使えます。

In [ ]:
plt.figure(figsize=(8, 4))           # ← 横8インチ × 縦4インチ
plt.bar(products, sales, color="steelblue", width=0.6)
plt.title("商品別 売上")
plt.xlabel("商品")
plt.ylabel("売上 (円)")
plt.show()

## 5. pandas との合わせ技 — 集計結果を棒グラフに

ここからが本番です。第2章で習った **`groupby` + `sort_values`** の結果をそのまま棒グラフにします。

Excel で言えば「ピボットテーブル → 並べ替え → グラフ挿入」の3ステップを **1 つのセル** で書ききるイメージです。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/python-data-basics"
DATA = f"{BASE}/data/pandas"

df = pd.read_excel(f"{DATA}/sales_2026-01.xlsx", sheet_name="売上", parse_dates=["date"])

# 商品別の売上合計を多い順に
by_product = df.groupby("product_name")["amount"].sum().sort_values(ascending=False)
print(by_product)

In [ ]:
# Series の index が x 軸ラベル、値が棒の高さになる
plt.figure(figsize=(9, 4))
plt.bar(by_product.index, by_product.values, color="steelblue")
plt.title("2026年1月 商品別 売上合計")
plt.xlabel("商品")
plt.ylabel("売上合計 (円)")
plt.xticks(rotation=30)   # x 軸ラベルが重なるなら回転
plt.show()

**ポイント**:

- `groupby` の結果（Series）から **`.index`（行ラベル）** と **`.values`（値）** を取り出して `plt.bar` に渡す
- `plt.xticks(rotation=30)` で **x 軸ラベルを 30 度回転** できる（商品名が長くて重なるとき便利）

## 練習問題

上で読み込んだ売上データ `df` を使って、以下のグラフを描いてください。

1. **「日付ごとの売上合計」の棒グラフ** を描いてください。タイトルは「日次売上推移」、x 軸は「日付」、y 軸は「売上 (円)」。ヒント: `df.groupby("date")["amount"].sum()`。
2. **「商品別の取引件数」の横棒グラフ** を描いてください。**件数が多い順** に並べてから描画してください。ヒント: 件数は `.count()`。
3. 1 の棒グラフを、棒の色を `"orange"`、サイズを `figsize=(10, 4)` にして描き直してください。

In [ ]:
# ここに書いてみてください



<details>
<summary>答えを見る</summary>

```python
# 1. 日付ごとの売上合計
daily = df.groupby("date")["amount"].sum()

plt.figure(figsize=(10, 4))
plt.bar(daily.index, daily.values)
plt.title("日次売上推移")
plt.xlabel("日付")
plt.ylabel("売上 (円)")
plt.xticks(rotation=45)
plt.show()

# 2. 商品別の取引件数（多い順）の横棒
count_by_product = df.groupby("product_name")["amount"].count().sort_values()
# barh は下から上に伸びるので、ascending=True にしておくと最大が上に来る

plt.figure(figsize=(7, 4))
plt.barh(count_by_product.index, count_by_product.values, color="seagreen")
plt.title("商品別 取引件数")
plt.xlabel("件数")
plt.ylabel("商品")
plt.show()

# 3. 1 の色とサイズ変更版
daily = df.groupby("date")["amount"].sum()

plt.figure(figsize=(10, 4))
plt.bar(daily.index, daily.values, color="orange")
plt.title("日次売上推移")
plt.xlabel("日付")
plt.ylabel("売上 (円)")
plt.xticks(rotation=45)
plt.show()
```

</details>

## よくあるエラー

### 1. 日本語が「□□□」と表示される
→ `import japanize_matplotlib` を忘れている。冒頭で実行する。

### 2. `plt.bar` に Series を渡したら `TypeError`
```python
plt.bar(by_product)   # ❌ x と y の両方が必要
```
→ `plt.bar(by_product.index, by_product.values)` のように **x（ラベル）と y（値）を分けて** 渡す。

### 3. グラフが重なって表示される
→ 前のセルで `plt.show()` を書き忘れて続けて `plt.bar(...)` を呼ぶと、同じ図に重ね描きされる。**1 つのグラフごとに `plt.show()`** を書く。

### 4. x 軸ラベルが重なって読めない
→ `plt.xticks(rotation=30)` か `rotation=45` で回転、または `plt.barh` で横棒に切り替え。

### 5. `figsize` の指定が効かない
```python
plt.bar(...)
plt.figure(figsize=(10, 4))    # ❌ 順番が逆
```
→ **`plt.figure(figsize=...)` は `plt.bar(...)` より前** に書く。

## まとめ

- グラフは **`import matplotlib.pyplot as plt`** + **`import japanize_matplotlib`** から始める
- 棒グラフは **`plt.bar(x, y)` → `plt.show()`** の 2 行で描ける
- **`title` / `xlabel` / `ylabel`** で意味が伝わるグラフに
- 縦棒 `bar` / 横棒 `barh` を **項目名の長さ** で使い分け
- **`figure(figsize=...)`** は `bar` の **前** に書く
- **`groupby` + `sort_values`** の結果 (Series) を `.index` / `.values` で取り出して棒グラフに

次は **3-2「折れ線と散布図」** で、推移を見るのに使う折れ線グラフと、相関を見る散布図を扱います。